### EXPLORATORY DATA ANALYSIS

This notebook is a template for Project EmpowerED, focused on Philippine reading proficiency in 2024–2025 and its relationship with socioeconomic factors and location.

**Research questions:**
1. How does the reading proficiency distribution of Filipino students in SY 2024–2025 compare to the latest Southeast Asian benchmarks from PISA/SEA-PLM?
2. To what degree do household income and parental education correlate with standardized reading comprehension scores in the Philippines?
3. Is there a statistically significant difference in literacy outcomes between students from urban and rural public elementary schools?


In [ ]:
# 1. Setup imports and global settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


## 2. Load data

Use your local datasets from PISA 2022, SEA-PLM 2024, FLEMMS 2024, or any national survey files. Update the path variables below to point to your CSV or other formatted files.


In [ ]:
# Example data loading template
# Replace these file paths with your actual dataset locations.
pisa_path = "data/pisa_2022_reading.csv"
sealpm_path = "data/sea_plm_2024_reading.csv"
flemms_path = "data/flemms_2024_literacy.csv"

# Example loading logic
try:
    pisa_df = pd.read_csv(pisa_path)
    sea_df = pd.read_csv(sealpm_path)
    flemms_df = pd.read_csv(flemms_path)
except FileNotFoundError:
    print("Update the file paths in the notebook to match your dataset locations.")
    pisa_df = pd.DataFrame()
    sea_df = pd.DataFrame()
    flemms_df = pd.DataFrame()

# Quick preview if data is loaded
for name, df in [("PISA", pisa_df), ("SEA-PLM", sea_df), ("FLEMMS", flemms_df)]:
    if not df.empty:
        print(f"{name} shape:", df.shape)
        display(df.head())


## 3. Data preparation and harmonization

Key tasks:
- Standardize score and proficiency level column names
- Create comparable variables for reading scores
- Map urban/rural location labels to a consistent format
- Create socioeconomic indicators for income and parental education


In [ ]:
# Data preparation helper functions

def prepare_reading_scores(df, score_col, country_col=None, region_col=None, urban_rural_col=None, income_col=None, parental_education_col=None):
    df = df.copy()
    mapping = {}
    if score_col:
        mapping[score_col] = "reading_score"
    if country_col:
        mapping[country_col] = "country"
    if region_col:
        mapping[region_col] = "region"
    if urban_rural_col:
        mapping[urban_rural_col] = "urban_rural"
    if income_col:
        mapping[income_col] = "income"
    if parental_education_col:
        mapping[parental_education_col] = "parental_education"
    df = df.rename(columns=mapping)
    if "urban_rural" in df.columns:
        df["urban_rural"] = df["urban_rural"].astype(str).str.title()
    return df

# Example normalization (update column names to match your actual dataset)
# pisa_df = prepare_reading_scores(pisa_df, score_col="READ_SCORE", country_col="CNTRY", region_col="REGION", urban_rural_col="URBAN_RURAL", income_col="INCOME", parental_education_col="PARED")
# sea_df = prepare_reading_scores(sea_df, score_col="READING_SCORE", country_col="COUNTRY", region_col="PROVINCE", urban_rural_col="SCHOOL_LOCATION", income_col="WEALTH_INDEX", parental_education_col="PARED")
# flemms_df = prepare_reading_scores(flemms_df, score_col="LIT_SCORE", region_col="REGION", urban_rural_col="AREA_TYPE")


## 4. RQ1: Compare Filipino reading proficiency to Southeast Asian benchmarks

Analysis steps:
- Plot score distributions for the Philippines and regional peers
- Compare proficiency achievement rates across countries
- Highlight the Philippines against SEA-PLM and PISA benchmark thresholds


In [ ]:
# RQ1 example visualizations

def plot_score_distribution(df, country_col="country", score_col="reading_score", countries=None, title=None):
    if countries is None:
        countries = df[country_col].dropna().unique().tolist()
    subset = df[df[country_col].isin(countries)]
    sns.kdeplot(data=subset, x=score_col, hue=country_col, common_norm=False, fill=True, alpha=0.4)
    plt.title(title or "Reading Score Distribution")
    plt.xlabel("Reading Score")
    plt.ylabel("Density")
    plt.legend(title="Country")
    plt.show()

# Example usage:
# benchmark_countries = ["Philippines", "Thailand", "Vietnam", "Indonesia", "Malaysia", "Singapore"]
# plot_score_distribution(sea_df, country_col="country", score_col="reading_score", countries=benchmark_countries, title="SEA-PLM Reading Score Distributions")

# Compare proficiency bands if available
# def plot_proficiency_bands(df, country_col="country", band_col="proficiency_band"):
#     band_counts = df.groupby([country_col, band_col]).size().reset_index(name="count")
#     band_pct = band_counts.groupby(country_col).apply(lambda x: x.assign(pct=x["count"] / x["count"].sum() * 100)).reset_index(drop=True)
#     sns.barplot(data=band_pct, x=country_col, y="pct", hue=band_col)
#     plt.title("Proficiency Band Share by Country")
#     plt.ylabel("Percent")
#     plt.show()

# Example statistics summary
# print("Philippines mean reading score:", sea_df.loc[sea_df["country"] == "Philippines", "reading_score"].mean())
# print("Regional mean reading score:", sea_df.groupby("country")["reading_score"].mean())


## 5. RQ2: Correlation of household income and parental education with reading scores

Analysis steps:
- Compute Pearson correlation between reading score, income, and parental education
- Visualize with scatter plots and regression lines
- Use boxplots to compare score distributions across wealth or education groups


In [ ]:
# RQ2 example correlation and visualization

def correlation_matrix(df, cols):
    corr = df[cols].corr(method="pearson")
    display(corr)
    sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
    plt.title("Correlation matrix")
    plt.show()

# Example usage:
# correlation_matrix(sea_df, ["reading_score", "income", "parental_education"])

def scatter_with_regression(df, x_col, y_col, hue_col=None, title=None):
    sns.lmplot(data=df, x=x_col, y=y_col, hue=hue_col, aspect=1.4, scatter_kws={"alpha":0.4})
    plt.title(title or f"{y_col} vs {x_col}")
    plt.show()

# Example usage:
# scatter_with_regression(sea_df, x_col="income", y_col="reading_score", hue_col="country", title="Reading Score vs Income")

# Boxplot by income or parental education group
# sns.boxplot(data=sea_df, x="income_quintile", y="reading_score")
# plt.title("Reading Score by Income Quintile")
# plt.show()


## 6. RQ3: Urban vs Rural literacy outcomes

Analysis steps:
- Compare reading score distributions for urban and rural schools
- Visualize differences with violin plots or boxplots
- Conduct a t-test or ANOVA to assess statistical significance
- Optionally map regional literacy rates from FLEMMS


In [ ]:
# RQ3 example analysis

def compare_urban_rural(df, score_col="reading_score", location_col="urban_rural"):
    sns.violinplot(data=df, x=location_col, y=score_col)
    plt.title("Reading Score Distribution by Urban/Rural")
    plt.show()

    groups = [group[score_col].dropna() for _, group in df.groupby(location_col)]
    if len(groups) == 2:
        t_stat, p_value = stats.ttest_ind(groups[0], groups[1], equal_var=False)
        print(f"t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
        if p_value < 0.05:
            print("There is a statistically significant difference between urban and rural reading scores.")
        else:
            print("No statistically significant difference found at alpha=0.05.")
    else:
        print("Urban/rural comparison requires exactly two groups, found:", df[location_col].unique())

# Example usage:
# compare_urban_rural(sea_df, score_col="reading_score", location_col="urban_rural")

# Optional: regional literacy mapping with FLEMMS data
# flemms_map = flemms_df.groupby("region")["functional_literacy_rate"].mean().reset_index()
# display(flemms_map)


## 7. Next steps and workflow

1. Load and inspect your local files: PISA 2022, SEA-PLM 2024, and FLEMMS 2024.
2. Clean and harmonize variable names in each dataset.
3. Use RQ1 section to compare the Philippines against SEA peers with distribution plots.
4. Use RQ2 section to measure socioeconomic correlations and visualize trends.
5. Use RQ3 section to test urban/rural differences and optionally map literacy rates.
6. Report findings with supporting charts and test statistics.

> Note: If you only have one dataset available, focus on the relevant section and label the comparison as local versus benchmark expectations.
